### 1. Set up env and access api keys to test


In [35]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

load_dotenv()

True

In [36]:
openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
alpaca_key = os.environ.get("ALPACA_API_KEY")
alpaca_secret = os.environ.get("ALPACA_SECRET_KEY")
alpaca_endpoint = os.environ.get("ALPACA_PAPER_ENDPOINT")

In [37]:
# Connect to client libraries

openai = OpenAI()

openrouter_url = "https://openrouter.ai/api/v1"

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


### 2. Set up models dict


In [38]:
JUDGE_MODEL = "gpt-5"
JUDGE_CLIENT = openai

models = [
    "gpt-5.5",
    "moonshotai/kimi-k2.6",
    "openai/gpt-oss-120b:free",
    "openai/gpt-oss-20b:free",
    "deepseek/deepseek-v4-pro",
    "anthropic/claude-sonnet-4.6",
]

clients = {
    "gpt-5.5": openai,
    "moonshotai/kimi-k2.6": openrouter,
    "openai/gpt-oss-120b:free": openrouter,
    "openai/gpt-oss-20b:free": openrouter,
    "deepseek/deepseek-v4-pro": openrouter,
    "anthropic/claude-sonnet-4.6": openrouter,
}


In [39]:
REASONING_MODELS = {
    "moonshotai/kimi-k2.6",
    "openai/gpt-oss-120b:free",
    "openai/gpt-oss-20b:free",
    "deepseek/deepseek-v4-pro",
}

### 3. Write the master code-generation prompt


In [ ]:
MASTER_PROMPT = """Write a complete Python script for Alpaca paper trading only. Return Python code only with no markdown fences and no commentary.\n
\n
Requirements:\n
- Trade only AAPL.\n
- Use real Alpaca market data via the latest trade endpoint specifically.\n
- Fetch the latest AAPL trade price.\n
- Read credentials from .env using these exact variable names: OPENROUTER_API_KEY, ALPACA_API_KEY, ALPACA_SECRET_KEY, ALPACA_PAPER_ENDPOINT (with the value https://paper-api.alpaca.markets/v2).\n
- Use Alpaca paper trading only. Never use live trading endpoints or live trading credentials.\n
- Submit at most one paper order per run.\n
- If the latest AAPL price is below 280, check Alpaca account buying power and buy exactly 1 share only if buying power is sufficient. Otherwise skip and print the reason.\n
- If the latest AAPL price is above 290, check the current AAPL position and sell exactly 1 share only if at least 1 share is already held. Otherwise skip and print the reason. Never short.\n
- If the latest AAPL price is between 280 and 290 inclusive, do nothing and print the reason.\n
- If the US market is closed, still proceed using the last available trade price.\n
- Print the latest trade price, decision made, precondition checks, whether an order was submitted, and the order response or error message.\n
- Handle missing credentials gracefully.\n
- Handle Alpaca API errors gracefully.\n
- Avoid infinite loops or repeated polling.\n
- Avoid hardcoded API keys.\n
- Avoid portfolio-wide logic.\n
- Avoid any ticker except AAPL.\n
"""


### 4. Generate code from one model


In [41]:
def _strip_code_fences(text):
    text = text.strip()
    if not text.startswith("```"):
        return text

    lines = text.splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


def generate_code(model_id, prompt):
    """Generate source code for one selected model.

    Args:
        model_id: Model identifier from the notebook's model list.
        prompt: Master prompt sent as the user message.

    Returns:
        Plain Python source with markdown fences removed.

    Raises:
        gr.Error: If the model is unknown, the API call fails, or the response is empty.
    """
    client = clients.get(model_id)
    if client is None:
        raise gr.Error(f"Unknown model: {model_id}")

    kwargs = {
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
    }

    if model_id in REASONING_MODELS:
        kwargs["extra_body"] = {"reasoning": {"effort": "high"}}

    try:
        response = client.chat.completions.create(**kwargs)
        content = response.choices[0].message.content or ""
    except Exception as exc:
        raise gr.Error(f"Code generation failed for {model_id}: {exc}") from exc

    if not content.strip():
        raise gr.Error(f"Model returned empty output for {model_id}.")

    return _strip_code_fences(content)


In [43]:
generated_model_code = generate_code(
    model_id="deepseek/deepseek-v4-pro", prompt=MASTER_PROMPT
)

### 5. Save generated scripts


In [44]:
def _safe_model_label(model_label):
    cleaned = [char.lower() if char.isalnum() else "_" for char in model_label.strip()]
    # isalnum checks with a string contains only alphanumeric characters, basically it removes everything other than alphabets and numbers
    safe_label = "".join(cleaned).strip("_")
    return safe_label or "model"


def save_generated_script(model_label, code, output_dir="outputs"):
    """Save generated model code to a predictable Python file.

    Args:
        model_label: Display label used to build the output filename.
        code: Generated Python source to save.
        output_dir: Folder where generated scripts are written.

    Returns:
        Relative path to the saved script.

    Raises:
        gr.Error: If the generated code is empty.
    """
    if not code.strip():
        raise gr.Error("Generated code is empty; nothing to save.")

    os.makedirs(output_dir, exist_ok=True)
    filename = f"generated_{_safe_model_label(model_label)}.py"
    file_path = os.path.join(output_dir, filename)

    with open(file_path, "w", encoding="utf-8") as handle:
        handle.write(code)

    return file_path


In [46]:
save_generated_script(model_label="deepseek/deepseek-v4-pro", code=generated_model_code)

'outputs/generated_deepseek_deepseek_v4_pro.py'